[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/reflections/kaleidoscope.ipynb)

# Two mirrors

A kaleidoscope is two mirrors meeting at an angle, and whatever lies between them seen again in every reflection. A mirror through the origin is a unit vector normal to it, and reflecting a point in the mirror is the sandwich of the point with that vector, negated. The product of the two mirror vectors is a rotor, which turns by twice the angle between the mirrors: with the mirrors at a half turn divided by $n$, it turns by $2\pi / n$, and $n$ of them make a scalar, which turns nothing.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import numpy as np
from IPython.display import Image, display

from numga import Algebra, NumpyContext
from examples.animation import save_animation
from examples.math.reflections import render

# The plane, and kaleidoscopes of three to six panels: mirrors meeting at a half turn divided by each.
ga = Algebra("x+y+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
PANELS = np.array([3, 4, 5, 6])
RESOLUTION = 320
FRAMES = 90
FRAME_MS = 60

## 1. The mirrors and their product

The first mirror lies along $x$, so its vector is $y$. The second mirror's vector is turned from the first by a half turn less the angle between the mirrors, so that both vectors point out of the wedge between them, and the two pair to $-\cos(\pi / n)$. Their product is the turn by twice the angle; its powers, up to the $n$-th, are the $n$ rotations of the kaleidoscope.

In [ ]:
first = mv.y                                                                    # [] Vector
second = ((mv.x ^ mv.y) * (-(np.pi - np.pi / PANELS) / 2)).exp() >> first     # [cases] Vector
pairing = first | second                                                        # [cases] Scalar
turn = second * first                                                           # [cases] Rotor
powers = turn.broadcast_to((PANELS.max(),) + turn.shape).cumprod(axis=0)       # [panels, cases] Rotor

In [ ]:
for case, panels in enumerate(PANELS):
    print(f"{panels} panels, the turn and its power of {panels}:")
    print(turn[case].coefficient_table())
    print(powers[panels - 1, case].coefficient_table())

## 2. Folding the plane into the wedge

A point lies on the wrong side of a mirror where it pairs negatively with the mirror's vector, and there it is reflected; elsewhere it stays. Reflecting in the two mirrors in turn, as many times as there are panels, brings every point of the plane into the wedge. Points that fold onto the same point of the wedge see the same thing, which is what the kaleidoscope shows.

In [ ]:
def fold(points: Vector, mirrors: tuple[Vector, ...], rounds: int) -> Vector:
    """The points, reflected round after round in each mirror they lie on the wrong side of."""
    for _ in range(rounds):
        for mirror in mirrors:
            side = mirror | points                                              # [...] Scalar
            # One on the wrong side of the mirror, zero on the right side.
            wrong = (side.abs() - side) / (2 * side.abs())                      # [...] Scalar
            points = points + wrong * (-(mirror >> points) - points)
    return points


u, v = render.disk(RESOLUTION)
points = mv.x * u + mv.y * v                                                    # [rows, columns] Vector
folded = fold(points, (first, second[:, None, None]), PANELS.max())            # [cases, rows, columns] Vector

## 3. The kaleidoscope

Beads tumble across the plane; each pixel shows the bead at its folded point, if any.

In [ ]:
display(Image(filename=save_animation(render.animate_kaleidoscope(folded, FRAMES), "kaleidoscope", FRAME_MS)))

In [ ]:
# checks
# The mirror vectors pair to -cos(pi / n); the n-th power of the turn is a scalar of size one; every
# folded point lies on the right side of both mirrors.
np.testing.assert_allclose(pairing.to_array(), -np.cos(np.pi / PANELS), atol=1e-12)
full = powers.kernel[PANELS - 1, np.arange(len(PANELS))]                       # [cases, 2]
np.testing.assert_allclose(np.abs(full[:, 0]), 1.0, atol=1e-12)
np.testing.assert_allclose(full[:, 1], 0.0, atol=1e-12)
for mirror in (first, second[:, None, None]):
    assert (mirror | folded).to_array().min() > -1e-12